# Featuresmith Tutorial: Getting Started

Welcome to Featuresmith! This notebook will guide you through the basics of loading tabular data, profiling it for statistical descriptors, and analyzing it using the built-in deterministic validation and leakage rule engine.

## Objectives
1. Install and load Featuresmith.
2. Load datasets from CSV, Parquet, or pandas DataFrames.
3. Run deterministic profiling to collect columns and row metrics.
4. Execute the rule engine to inspect findings.

---

### Step 1: Install Featuresmith

Run `pip install featuresmith-core` in your command line. Let's import featuresmith and print the version.

In [ ]:
import featuresmith as fs

print(
    f"Featuresmith version: {fs.__version__ if hasattr(fs, '__version__') else '0.1.0'}"
)

### Step 2: Load a Dataset

Featuresmith can load files from CSV, Parquet, Excel, or wrap in-memory pandas or Polars DataFrames into a normalized `Dataset` object. Let's load the clean **Iris** dataset.

In [ ]:
import os

# Ensure you have run python examples/download_datasets.py first!
dataset_path = os.path.join("..", "data", "processed", "iris.csv")

dataset = fs.load(dataset_path)
print("Dataset loaded successfully.")
print(f"Number of rows: {dataset.row_count}")
print(f"Schema names : {dataset.schema.names}")

### Step 3: Profile the Data

The profiling engine runs vectorized computations to compile summaries without formatting layout charts. This returns a typed, serializable `ProfileResult`.

In [ ]:
profile = fs.profile(dataset)

print("Column categories count:")
print(f"- Column count: {profile.dataset_summary.column_count}")
print(f"- Missing value rate: {profile.dataset_summary.missing_percentage:.2f}%")

print("\nDetailed column types:")
for col, col_prof in profile.column_profiles.items():
    print(
        f"- {col}: Type={col_prof.logical_type}, Missing count={col_prof.missing_count}"
    )

### Step 4: Run the Rule Engine

The rule engine checks the `ProfileResult` against 8 deterministic rules to isolate data quality and target leakage findings. Let's analyze the Iris dataset (which should be completely clean).

In [ ]:
result = fs.analyze(dataset)

print(f"Audit completed in {result.execution_time_ms:.2f} ms.")
print(f"Executed rules count: {len(result.executed_rules)}")
print(f"Total findings: {len(result.findings)}")

if len(result.findings) == 0:
    print("\nSuccess! The Iris dataset contains no rule violations or quality issues.")

### Step 5: Serialize Findings

All results are `frozen` dataclasses, which makes them easy to serialize to standard JSON/dictionaries for storage or logging.

In [ ]:
import json

report_json = json.dumps(result.to_dict(), indent=2, default=str)
print("Preview of serialized result structure:")
print("\n".join(report_json.splitlines()[:15]))